In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Imports ===

%matplotlib inline

import os
import sys
import json
import random
import platform
import re
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from pprint import pprint

import numpy as np
import pandas as pd
import yaml

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import ultralytics
from ultralytics import YOLO

from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

In [ ]:
# === Cell 3: Config ===

# --- Project paths ---
EXPORT_ROOT    = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export")
TRAIN_ROOT     = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260603_Yolo_Deployment_Training")
BEST_HPO       = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260528_Yolo_Hyperparameter_Tuning_yolov8n/results/20260528_Yolo_Hyperparameter_Tuning_yolov8n_best_trial_hyperparameters.csv")
HPO_ROOT       = BEST_HPO.parents[1]

IMAGES_DIR = EXPORT_ROOT / "images" / "all"
LABELS_DIR = EXPORT_ROOT / "labels" / "all"

OBJECT_SPLIT_CSV = EXPORT_ROOT / "object_split.csv"
METADATA_AGG_CSV = EXPORT_ROOT / "metadata_aggregated.csv"
BASE_DATA_YAML   = EXPORT_ROOT / "data.yaml"

# --- Output folders ---
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS_DIR    = TRAIN_ROOT / "splits"
RUNS_DIR      = TRAIN_ROOT / "runs"
RESULTS_DIR   = TRAIN_ROOT / "results"

for p in [SPLITS_DIR, RUNS_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_NAME    = TRAIN_ROOT.name
EXPERIMENT_NAME = "DeploymentSegTraining"
RUN_NAME        = f"{PROJECT_NAME}_{EXPERIMENT_NAME}"

# --- Dataset semantics ---
CLASS_MAP = {
    "body":  0,
    "cap":   1,
    "label": 2,
}
MATERIAL_CLASSES = ["HDPE", "PET", "PP", "PS"]

# --- Reproducibility and execution ---
RANDOM_SEED     = 42
DEVICE          = 0
WORKERS         = 2
ALLOW_OVERWRITE = True

# --- Model selection ---
SELECTED_MODEL = "yolov8n-seg.pt"
TASK           = "segment"

# --- Training setup ---
TRAIN_EPOCHS             = 300
TRAIN_PATIENCE           = 50
BATCH_SIZE               = 16
IMAGE_SIZE               = 640
IMAGE_RECT               = True
SAVE_PLOTS               = True
PRETRAINED               = True
VERBOSE                  = True
VALIDATE_DURING_TRAINING = True

# --- Object-level split (80/20 train/val, material-stratified) ---
VAL_FRACTION = 0.2    # fraction of objects used for validation

# --- HPO parsing ---
BEST_HPO_DF = pd.read_csv(BEST_HPO)
BEST_HPO_DF.columns = [str(c).strip() for c in BEST_HPO_DF.columns]

required_cols = {"trial_id", "hyperparameter", "value"}
missing_cols  = required_cols - set(BEST_HPO_DF.columns)
if missing_cols:
    raise ValueError(f"BEST_HPO missing columns: {missing_cols}")

BEST_TRIAL_ID = BEST_HPO_DF["trial_id"].iloc[0]

# Maps HPO CSV names (with or without underscores) to Ultralytics train() keys.
HPO_TO_YOLO_KEY_MAP = {
    "lr0":             "lr0",
    "lrf":             "lrf",
    "momentum":        "momentum",
    "weight_decay":    "weight_decay",
    "weightdecay":     "weight_decay",
    "warmup_epochs":   "warmup_epochs",
    "warmupepochs":    "warmup_epochs",
    "warmup_momentum": "warmup_momentum",
    "warmupmomentum":  "warmup_momentum",
    "box":             "box",
    "cls":             "cls",
    "dfl":             "dfl",
    "hsv_h":           "hsv_h",
    "hsvh":            "hsv_h",
    "hsv_s":           "hsv_s",
    "hsvs":            "hsv_s",
    "hsv_v":           "hsv_v",
    "hsvv":            "hsv_v",
    "degrees":         "degrees",
    "translate":       "translate",
    "scale":           "scale",
    "shear":           "shear",
    "perspective":     "perspective",
    "flipud":          "flipud",
    "fliplr":          "fliplr",
    "bgr":             "bgr",
    "mosaic":          "mosaic",
    "mixup":           "mixup",
    "copy_paste":      "copy_paste",
    "copypaste":       "copy_paste",
    "cutmix":          "cutmix",
}

VALID_YOLO_HPARAMS = {
    "lr0", "lrf", "momentum", "weight_decay", "warmup_epochs", "warmup_momentum",
    "box", "cls", "dfl", "hsv_h", "hsv_s", "hsv_v", "degrees", "translate",
    "scale", "shear", "perspective", "flipud", "fliplr", "bgr", "mosaic",
    "mixup", "copy_paste", "cutmix",
}

# Converts CSV strings to int, float or bool where possible.
def normalize_scalar(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, str):
        value = value.strip()
        try:
            f = float(value)
            return int(f) if f.is_integer() else f
        except Exception:
            if value.lower() in {"true", "false"}:
                return value.lower() == "true"
            return value
    return value

def build_best_hpo_dict(hpo_df):
    best_dict = {}
    ignored   = []
    for _, row in hpo_df.iterrows():
        raw_key    = str(row["hyperparameter"]).strip()
        raw_value  = normalize_scalar(row["value"])
        mapped_key = HPO_TO_YOLO_KEY_MAP.get(raw_key)
        if mapped_key is None or mapped_key not in VALID_YOLO_HPARAMS:
            ignored.append(raw_key)
            continue
        best_dict[mapped_key] = raw_value
    return best_dict, sorted(set(ignored))

BEST_HPO_HPARAMS, IGNORED_HPO_KEYS = build_best_hpo_dict(BEST_HPO_DF)

# Optional overrides applied on top of the best HPO trial.
MANUAL_HPARAM_OVERRIDES = {
    # "lr0": 0.001,
    # "mosaic": 0.5,
}

FINAL_TRAIN_HPARAMS = dict(BEST_HPO_HPARAMS)
FINAL_TRAIN_HPARAMS.update(MANUAL_HPARAM_OVERRIDES)

# --- Export paths ---
FINAL_HPARAM_CSV        = RESULTS_DIR / f"{PROJECT_NAME}_final_hyperparameters.csv"
FINAL_TRAIN_KWARGS_JSON = RESULTS_DIR / f"{PROJECT_NAME}_final_train_kwargs.json"
FINAL_RUN_SUMMARY_JSON  = RESULTS_DIR / f"{PROJECT_NAME}_final_run_summary.json"
FINAL_EVAL_CSV          = RESULTS_DIR / f"{PROJECT_NAME}_final_val_metrics.csv"

# --- Deterministic seeds ---
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Configuration")
print("-" * 80)
print("EXPORT_ROOT:                ", EXPORT_ROOT)
print("TRAIN_ROOT:                 ", TRAIN_ROOT)
print("HPO_ROOT:                   ", HPO_ROOT)
print("BEST_HPO:                   ", BEST_HPO)
print("SELECTED_MODEL:             ", SELECTED_MODEL)
print("RUN_NAME:                   ", RUN_NAME)
print("DEVICE:                     ", DEVICE)
print("VAL_FRACTION:               ", VAL_FRACTION)
print("BEST_TRIAL_ID:              ", BEST_TRIAL_ID)
print("IGNORED_HPO_KEYS:           ", IGNORED_HPO_KEYS)
print("FINAL_TRAIN_HPARAMS:")
pprint(FINAL_TRAIN_HPARAMS)

pd.DataFrame([
    {"hyperparameter": k, "value": v} for k, v in FINAL_TRAIN_HPARAMS.items()
]).sort_values("hyperparameter").to_csv(FINAL_HPARAM_CSV, index=False)

print("\nSaved final hyperparameters to:", FINAL_HPARAM_CSV)

In [ ]:
# === Cell 4: Sanity Check ===

print("Python:            ", sys.version)
print("Platform:          ", platform.platform())
print("torch version:     ", torch.__version__)
print("CUDA available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count: ", torch.cuda.device_count())
    print("CUDA device 0:     ", torch.cuda.get_device_name(0))
print("ultralytics version:", ultralytics.__version__)

PATHS_TO_CHECK = [
    ("EXPORT_ROOT",      EXPORT_ROOT),
    ("TRAIN_ROOT",       TRAIN_ROOT),
    ("HPO_ROOT",         HPO_ROOT),
    ("IMAGES_DIR",       IMAGES_DIR),
    ("LABELS_DIR",       LABELS_DIR),
    ("OBJECT_SPLIT_CSV", OBJECT_SPLIT_CSV),
    ("METADATA_AGG_CSV", METADATA_AGG_CSV),
    ("BASE_DATA_YAML",   BASE_DATA_YAML),
    ("BEST_HPO",         BEST_HPO),
    ("SPLITS_DIR",       SPLITS_DIR),
    ("RUNS_DIR",         RUNS_DIR),
    ("RESULTS_DIR",      RESULTS_DIR),
]

all_paths_ok = True
print("\nPath checks")
for path_name, path_value in PATHS_TO_CHECK:
    exists = path_value.exists()
    kind   = "DIR " if path_value.is_dir() else "FILE"
    status = "OK     " if exists else "MISSING"
    print(f"  [{kind}] {status}  {path_name}: {path_value}")
    if path_name in {"EXPORT_ROOT", "IMAGES_DIR", "LABELS_DIR",
                     "OBJECT_SPLIT_CSV", "BASE_DATA_YAML", "BEST_HPO"} and not exists:
        all_paths_ok = False

if FINAL_TRAIN_KWARGS_JSON.exists() and not ALLOW_OVERWRITE:
    raise FileExistsError(
        f"Refusing to overwrite existing config: {FINAL_TRAIN_KWARGS_JSON}. "
        "Set ALLOW_OVERWRITE=True to continue."
    )

objectsplitdf = None
imagepaths    = list(IMAGES_DIR.glob("*")) if IMAGES_DIR.exists() else []

if OBJECT_SPLIT_CSV.exists():
    objectsplitdf = pd.read_csv(OBJECT_SPLIT_CSV, dtype={"object_id": str})
    objectsplitdf.columns = [c.strip() for c in objectsplitdf.columns]
    print("\nobject_split.csv columns:", list(objectsplitdf.columns))

    uniqueobjects = objectsplitdf.drop_duplicates(subset=["material", "object_id"])
    print("\nDataset stats")
    print("  Distinct objects:  ", len(uniqueobjects))
    print("  Distinct materials:", uniqueobjects["material"].nunique())
    print("  Total images:      ", len(imagepaths))
    display(uniqueobjects.groupby("material")["object_id"].nunique().rename("n_objects"))
else:
    raise FileNotFoundError(f"object_split.csv not found: {OBJECT_SPLIT_CSV}")

try:
    YOLO(SELECTED_MODEL)
    print(f"\nModel instantiation OK: {SELECTED_MODEL}")
except Exception as exc:
    raise RuntimeError(f"Model instantiation failed for {SELECTED_MODEL}") from exc

if not all_paths_ok:
    raise RuntimeError("One or more required paths are missing. Fix config before continuing.")

print("\nSanity check PASS")

In [ ]:
# === Cell 5: Filename Parsing ===

# Expected export filename format: HDPE_obj_01_per_0_2026-04-21.png
# Images are grouped by (material, object_id) for the object-level split.

FILENAME_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)"
    r"_obj_(?P<objectid>\d+)"
    r"_per_(?P<perspectiveid>\d+)"
    r"_(?P<date>\d{4}-\d{2}-\d{2})"
    r"\.(?P<extension>png|jpg|jpeg)$",
    re.IGNORECASE,
)

def parse_material_object_perspective(filename: str):
    match = FILENAME_REGEX.match(filename)
    if not match:
        raise ValueError(f"Filename does not match expected pattern: {filename}")
    material      = match.group("material")
    objectid      = match.group("objectid")
    perspectiveid = match.group("perspectiveid")
    datestr       = match.group("date")
    return material, objectid, perspectiveid, datestr

object_to_images = defaultdict(list)
bad_filenames    = []

for image_path in imagepaths:
    filename = image_path.name
    try:
        material, objectid, perspectiveid, datestr = parse_material_object_perspective(filename)
        object_to_images[(material, objectid)].append(image_path)
    except ValueError:
        bad_filenames.append(filename)

print("Mapped objects from filenames:", len(object_to_images))

if bad_filenames:
    print(f"\nWARNING: {len(bad_filenames)} filenames did not match expected pattern:")
    for bad_name in bad_filenames[:10]:
        print("  -", bad_name)
    if len(bad_filenames) > 10:
        print(f"  ... and {len(bad_filenames) - 10} more")

if objectsplitdf is not None:
    uniqueobjects   = objectsplitdf.drop_duplicates(subset=["material", "object_id"])
    split_objects   = set(zip(uniqueobjects["material"], uniqueobjects["object_id"]))
    mapped_objects  = set(object_to_images.keys())

    missing_in_images = split_objects - mapped_objects
    extra_in_images   = mapped_objects - split_objects

    print("\nConsistency checks")
    print("  Objects in object_split.csv but missing in images:", len(missing_in_images))
    print("  Objects in images but missing in object_split.csv:", len(extra_in_images))

    if missing_in_images:
        print("  Example missing:", list(missing_in_images)[:10])
    if extra_in_images:
        print("  Example extra:  ", list(extra_in_images)[:10])

    if len(missing_in_images) > 0:
        print("\nWARNING: some objects in the split CSV have no images — check IMAGES_DIR path.")
    else:
        print("\nFilename mapping OK — all split objects have images.")

In [ ]:
# === Cell 6: Object-Level Train/Val Split ===
# Uses all objects (ignoring the cv/test assignment in object_split.csv) with a random
# object-level train/val split per material (VAL_FRACTION). No test set is created.

random.seed(RANDOM_SEED)

# One row per object, regardless of its original split
all_objects_df = objectsplitdf.drop_duplicates(subset=["material", "object_id"]).copy()

def objects_to_image_paths(object_keys):
    paths = []
    for material, object_id in object_keys:
        images = object_to_images.get((material, object_id), [])
        if not images:
            print(f"WARNING: no images found for object {(material, object_id)}")
        paths.extend(images)
    return sorted(paths)

# --- Material-stratified object-level split ---
materials        = sorted(all_objects_df["material"].unique())
train_keys_all   = []
val_keys_all     = []
split_detail_rows = []

for material in materials:
    material_df  = all_objects_df[all_objects_df["material"] == material]
    object_ids   = sorted(material_df["object_id"].unique())
    random.shuffle(object_ids)

    num_val   = max(1, round(len(object_ids) * VAL_FRACTION))
    num_train = len(object_ids) - num_val

    val_ids   = object_ids[:num_val]
    train_ids = object_ids[num_val:]

    train_keys_local = [(material, oid) for oid in train_ids]
    val_keys_local   = [(material, oid) for oid in val_ids]

    train_keys_all.extend(train_keys_local)
    val_keys_all.extend(val_keys_local)

    split_detail_rows.append({
        "material":     material,
        "total_objects": len(object_ids),
        "train_objects": num_train,
        "train_images":  len(objects_to_image_paths(train_keys_local)),
        "val_objects":   num_val,
        "val_images":    len(objects_to_image_paths(val_keys_local)),
    })

print("Deployment split summary (object-level 80/20, material-stratified)")
split_detail_df = pd.DataFrame(split_detail_rows)
total_row = pd.DataFrame([{
    "material":      "TOTAL",
    "total_objects":  split_detail_df["total_objects"].sum(),
    "train_objects":  split_detail_df["train_objects"].sum(),
    "train_images":   split_detail_df["train_images"].sum(),
    "val_objects":    split_detail_df["val_objects"].sum(),
    "val_images":     split_detail_df["val_images"].sum(),
}])
split_detail_df = pd.concat([split_detail_df, total_row], ignore_index=True)
display(split_detail_df)

# --- Write train/val split files ---
train_images = objects_to_image_paths(train_keys_all)
val_images   = objects_to_image_paths(val_keys_all)

DEPLOY_TRAIN_TXT = SPLITS_DIR / "deploy_train.txt"
DEPLOY_VAL_TXT   = SPLITS_DIR / "deploy_val.txt"

with DEPLOY_TRAIN_TXT.open("w", encoding="utf-8") as f:
    for p in train_images:
        f.write(str(p.resolve()) + "\n")

with DEPLOY_VAL_TXT.open("w", encoding="utf-8") as f:
    for p in val_images:
        f.write(str(p.resolve()) + "\n")

# --- Write data.yaml ---
with open(BASE_DATA_YAML, "r", encoding="utf-8") as f:
    base_data = yaml.safe_load(f)

DEPLOY_DATA_YAML = SPLITS_DIR / "data_deploy.yaml"

deploy_data          = dict(base_data)
deploy_data["path"]  = str(EXPORT_ROOT.resolve())
deploy_data["train"] = str(DEPLOY_TRAIN_TXT.resolve())
deploy_data["val"]   = str(DEPLOY_VAL_TXT.resolve())
# No test split in the deployment YAML
deploy_data.pop("test", None)

with DEPLOY_DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(deploy_data, f, sort_keys=False)

# --- Save split assignment CSV ---
split_assignment_rows = []
for material, oid in train_keys_all:
    split_assignment_rows.append({"material": material, "object_id": oid, "split": "train"})
for material, oid in val_keys_all:
    split_assignment_rows.append({"material": material, "object_id": oid, "split": "val"})

split_assignment_df = pd.DataFrame(split_assignment_rows).sort_values(
    ["material", "split", "object_id"]
).reset_index(drop=True)

SPLIT_ASSIGNMENT_CSV = RESULTS_DIR / f"{PROJECT_NAME}_split_assignments.csv"
split_assignment_df.to_csv(SPLIT_ASSIGNMENT_CSV, index=False)

print("\nSplit files")
print("DEPLOY_TRAIN_TXT: ", DEPLOY_TRAIN_TXT)
print("DEPLOY_VAL_TXT:   ", DEPLOY_VAL_TXT)
print("DEPLOY_DATA_YAML: ", DEPLOY_DATA_YAML)
print("SPLIT_ASSIGNMENT: ", SPLIT_ASSIGNMENT_CSV)
print("#train images:    ", len(train_images))
print("#val images:      ", len(val_images))
print("#total images:    ", len(train_images) + len(val_images))

In [ ]:
# === Cell 7: Deployment Training ===

# Splits e.g. "yolov8n-seg.pt" into ("yolov8", "n").
def parse_model_version_size(model_name: str):
    stem = Path(model_name).stem
    base = stem.split("-")[0]
    if not base.startswith("yolo") or len(base) < 6:
        return base, None
    return base[:-1], base[-1]

def serialize_for_json(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, dict):
        return {k: serialize_for_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [serialize_for_json(v) for v in obj]
    return obj

MODEL_VERSION, MODEL_SIZE = parse_model_version_size(SELECTED_MODEL)

train_kwargs = {
    "data": str(DEPLOY_DATA_YAML),
    "epochs": TRAIN_EPOCHS,
    "patience": TRAIN_PATIENCE,
    "batch": BATCH_SIZE,
    "imgsz": IMAGE_SIZE,
    "device": DEVICE,
    "workers": WORKERS,
    "project": str(RUNS_DIR),
    "name": RUN_NAME,
    "exist_ok": ALLOW_OVERWRITE,
    "task": TASK,
    "rect": IMAGE_RECT,
    "seed": RANDOM_SEED,
    "deterministic": True,
    "val": VALIDATE_DURING_TRAINING,
    "plots": SAVE_PLOTS,
    "pretrained": PRETRAINED,
    "verbose": VERBOSE,
    "optimizer": "AdamW",   # explicit optimizer so the HPO lr0 and momentum are used
}
# Best HPO hyperparameters are added on top of the base settings.
train_kwargs.update(FINAL_TRAIN_HPARAMS)

with FINAL_TRAIN_KWARGS_JSON.open("w", encoding="utf-8") as f:
    json.dump(serialize_for_json(train_kwargs), f, indent=2, ensure_ascii=False)

print("Training kwargs")
pprint(train_kwargs)

deploy_model   = YOLO(SELECTED_MODEL)
train_results  = deploy_model.train(**train_kwargs)

# Run directory from the train results; falls back to RUNS_DIR / RUN_NAME.
if train_results is not None and hasattr(train_results, "save_dir"):
    DEPLOY_RUN_DIR = Path(train_results.save_dir)
elif train_results is not None and hasattr(train_results, "savedir"):
    DEPLOY_RUN_DIR = Path(train_results.savedir)
else:
    DEPLOY_RUN_DIR = RUNS_DIR / RUN_NAME

DEPLOY_WEIGHTS_BEST = DEPLOY_RUN_DIR / "weights" / "best.pt"
DEPLOY_WEIGHTS_LAST = DEPLOY_RUN_DIR / "weights" / "last.pt"
DEPLOY_RESULTS_CSV  = DEPLOY_RUN_DIR / "results.csv"

run_summary = {
    "timestamp":        datetime.now().isoformat(),
    "project_name":     PROJECT_NAME,
    "experiment_name":  EXPERIMENT_NAME,
    "run_name":         RUN_NAME,
    "selected_model":   SELECTED_MODEL,
    "model_version":    MODEL_VERSION,
    "model_size":       MODEL_SIZE,
    "best_trial_id":    int(BEST_TRIAL_ID) if pd.notna(BEST_TRIAL_ID) else None,
    "val_fraction":     VAL_FRACTION,
    "num_train_images": len(train_images),
    "num_val_images":   len(val_images),
    "deploy_data_yaml": DEPLOY_DATA_YAML,
    "deploy_run_dir":   DEPLOY_RUN_DIR,
    "weights_best":     DEPLOY_WEIGHTS_BEST,
    "weights_last":     DEPLOY_WEIGHTS_LAST,
    "results_csv":      DEPLOY_RESULTS_CSV,
    "train_kwargs":     train_kwargs,
}

with FINAL_RUN_SUMMARY_JSON.open("w", encoding="utf-8") as f:
    json.dump(serialize_for_json(run_summary), f, indent=2, ensure_ascii=False)

print("Training finished")
print("DEPLOY_RUN_DIR:      ", DEPLOY_RUN_DIR)
print("DEPLOY_WEIGHTS_BEST: ", DEPLOY_WEIGHTS_BEST, DEPLOY_WEIGHTS_BEST.exists())
print("DEPLOY_RESULTS_CSV:  ", DEPLOY_RESULTS_CSV, DEPLOY_RESULTS_CSV.exists())
print("Run summary saved:   ", FINAL_RUN_SUMMARY_JSON)

In [ ]:
# === Cell 8: Training Curves and Val Summary ===

if "DEPLOY_WEIGHTS_BEST" not in globals() or not DEPLOY_WEIGHTS_BEST.exists():
    raise FileNotFoundError("Best weights not found. Run Cell 7 first.")

if "DEPLOY_RUN_DIR" not in globals():
    DEPLOY_RUN_DIR = RUNS_DIR / RUN_NAME

DEPLOY_RESULTS_CSV = DEPLOY_RUN_DIR / "results.csv"

def _find_col(df, patterns):
    """First column whose name contains any pattern (case-insensitive)."""
    for p in patterns:
        for c in df.columns:
            if p.lower() in c.lower():
                return c
    return None

# --- Training curves from Ultralytics results.csv ---
if not DEPLOY_RESULTS_CSV.exists():
    print(f"WARNING: results.csv not found at {DEPLOY_RESULTS_CSV} — skipping curves.")
else:
    tdf = pd.read_csv(DEPLOY_RESULTS_CSV)
    tdf.columns = [c.strip() for c in tdf.columns]
    ep  = pd.to_numeric(tdf[_find_col(tdf, ["epoch"]) or tdf.columns[0]], errors="coerce")

    # --- Loss curves ---
    loss_specs = [
        ("Train box loss",  ["train/box", "box_loss"]),
        ("Train seg loss",  ["train/seg", "seg_loss"]),
        ("Train cls loss",  ["train/cls", "cls_loss"]),
        ("Train dfl loss",  ["train/dfl", "dfl_loss"]),
        ("Val box loss",    ["val/box"]),
        ("Val seg loss",    ["val/seg"]),
        ("Val cls loss",    ["val/cls"]),
        ("Val dfl loss",    ["val/dfl"]),
    ]
    fig_l, axes_l = plt.subplots(2, 4, figsize=(18, 7))
    fig_l.suptitle(f"Loss curves — {RUN_NAME}", fontsize=12)
    for ax, (lbl, pats) in zip(axes_l.reshape(-1), loss_specs):
        col = _find_col(tdf, pats)
        clr = "tab:blue" if "Train" in lbl else "tab:orange"
        if col:
            ax.plot(ep, pd.to_numeric(tdf[col], errors="coerce"), color=clr)
            ax.set_title(lbl, fontsize=9)
            ax.set_xlabel("Epoch", fontsize=8)
            ax.grid(alpha=0.3)
        else:
            ax.set_title(f"{lbl}\n(not found)", fontsize=9)
            ax.axis("off")
    plt.tight_layout()
    loss_png = RESULTS_DIR / f"{PROJECT_NAME}_loss_curves.png"
    fig_l.savefig(loss_png, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", loss_png)

    # --- Validation metric curves ---
    metric_specs = [
        ("Precision",  ["precision(M)", "precision"]),
        ("Recall",     ["recall(M)",    "recall"]),
        ("mAP@50",     ["mAP50(M)",     "mAP50"]),
        ("mAP@50-95",  ["mAP50-95(M)", "mAP50-95"]),
    ]
    fig_m, axes_m = plt.subplots(1, 4, figsize=(18, 4))
    fig_m.suptitle(f"Validation metrics per epoch — {RUN_NAME}", fontsize=12)
    for ax, (lbl, pats) in zip(axes_m, metric_specs):
        col = _find_col(tdf, pats)
        if col:
            vals = pd.to_numeric(tdf[col], errors="coerce")
            ax.plot(ep, vals, color="tab:green")
            best_i = vals.idxmax()
            bv, be = vals[best_i], int(ep[best_i])
            ax.axvline(be, color="red", linestyle="--", alpha=0.6,
                       label=f"best={bv:.3f} @ep{be}")
            ax.set_title(lbl); ax.set_xlabel("Epoch")
            ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        else:
            ax.set_title(f"{lbl}\n(not found)"); ax.axis("off")
    plt.tight_layout()
    metric_png = RESULTS_DIR / f"{PROJECT_NAME}_metric_curves.png"
    fig_m.savefig(metric_png, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", metric_png)

# --- Best-epoch summary ---
summary = {
    "model":     SELECTED_MODEL,
    "run_name":  RUN_NAME,
    "weights":   str(DEPLOY_WEIGHTS_BEST),
    "num_train_images": len(train_images),
    "num_val_images":   len(val_images),
}

if DEPLOY_RESULTS_CSV.exists():
    for lbl, pats in metric_specs:
        col = _find_col(tdf, pats)
        if col:
            vals = pd.to_numeric(tdf[col], errors="coerce")
            key  = lbl.lower().replace(' ', '_').replace('@', '')
            summary[f"best_val_{key}"]       = round(float(vals.max()), 4)
            summary[f"best_val_{key}_epoch"] = int(ep[vals.idxmax()])

pd.DataFrame([summary]).to_csv(FINAL_EVAL_CSV, index=False)
print("\nSaved val summary CSV to:", FINAL_EVAL_CSV)

print("\nBest validation metrics:")
display(pd.DataFrame([{k: v for k, v in summary.items() if k.startswith("best_val_")}]))

print("\nDeployment weights ready at:")
print(" ", DEPLOY_WEIGHTS_BEST)

In [ ]:
# === Cell 9: Full-Dataset Inference and Region Metrics ===
# Predicts on all images, writes predicted YOLO labels and a timing summary, then
# computes per-region segmentation metrics per image and per object.

import cv2  # for polygon rasterisation (fillPoly)

# --- Inference config ---
INFER_CONF      = 0.50    # confidence threshold
INFER_IOU       = 0.50    # NMS IoU threshold
INFER_MAX_DET   = 300
INFER_RETINA    = True    # higher-quality masks

PRED_LABELS_DIR = TRAIN_ROOT / "labels"
PRED_LABELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Per-region metrics config ---
# IoU threshold for greedy matching (as for mAP50): matched pairs are TP,
# unmatched GT are FN, unmatched predictions are FP.
MATCH_IOU_THRESHOLD = 0.50

CLASS_ID_TO_NAME = {v: k for k, v in CLASS_MAP.items()}

PER_IMAGE_CSV  = RESULTS_DIR / f"{PROJECT_NAME}_per_image_region_metrics.csv"
PER_OBJECT_CSV = RESULTS_DIR / f"{PROJECT_NAME}_per_object_region_metrics.csv"

# --- Collect all images ---
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}
all_image_paths  = sorted(
    p for p in IMAGES_DIR.iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
)
print(f"Images to process: {len(all_image_paths)}")
print(f"Saving predicted labels to: {PRED_LABELS_DIR}")

# --- Resolve deployment weights ---
# Order: in-memory DEPLOY_WEIGHTS_BEST, weights_best from the run summary JSON,
# RUNS_DIR/RUN_NAME/weights/best.pt (fixed name since exist_ok=ALLOW_OVERWRITE),
# newest RUNS_DIR/RUN_NAME*/weights/best.pt.
weights_source = None

if "DEPLOY_WEIGHTS_BEST" in globals() and Path(DEPLOY_WEIGHTS_BEST).exists():
    DEPLOY_WEIGHTS_BEST = Path(DEPLOY_WEIGHTS_BEST)
    weights_source = "in-memory (Cell 7 ran this session)"

elif FINAL_RUN_SUMMARY_JSON.exists():
    with FINAL_RUN_SUMMARY_JSON.open("r", encoding="utf-8") as f:
        _run_summary = json.load(f)
    _candidate = Path(_run_summary.get("weights_best", ""))
    if _candidate.exists():
        DEPLOY_WEIGHTS_BEST = _candidate
        weights_source = f"run summary JSON: {FINAL_RUN_SUMMARY_JSON.name}"

if weights_source is None:
    _candidate = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
    if _candidate.exists():
        DEPLOY_WEIGHTS_BEST = _candidate
        weights_source = f"deterministic path: {RUN_NAME}/weights/best.pt"

if weights_source is None:
    # Newest best.pt among run folders starting with RUN_NAME
    _matches = sorted(
        RUNS_DIR.glob(f"{RUN_NAME}*/weights/best.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if _matches:
        DEPLOY_WEIGHTS_BEST = _matches[0]
        weights_source = f"most recent matching run folder: {DEPLOY_WEIGHTS_BEST.parent.parent.name}"

if weights_source is None:
    raise FileNotFoundError(
        f"Could not locate deployment weights automatically. Checked:\n"
        f"  - in-memory DEPLOY_WEIGHTS_BEST\n"
        f"  - {FINAL_RUN_SUMMARY_JSON}\n"
        f"  - {RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'}\n"
        f"  - {RUNS_DIR}/{RUN_NAME}*/weights/best.pt\n"
        f"Run Cell 7 first, or set DEPLOY_WEIGHTS_BEST manually."
    )

print(f"Using weights ({weights_source}):")
print(f"  {DEPLOY_WEIGHTS_BEST}")

infer_model = YOLO(str(DEPLOY_WEIGHTS_BEST))

# --- Inference loop (also collects detections for the region metrics) ---
infer_times_ms = []   # net inference ms per image (from result.speed)
skipped        = []

# Inputs for the region metrics, keyed by image stem
image_shapes   = {}   # {stem: (h, w)}
pred_instances = {}   # {stem: [{cls, conf, poly_norm}, ...]}

for image_path in all_image_paths:
    results = infer_model.predict(
        source      = str(image_path),
        conf        = INFER_CONF,
        iou         = INFER_IOU,
        imgsz       = IMAGE_SIZE,
        max_det     = INFER_MAX_DET,
        retina_masks= INFER_RETINA,
        verbose     = False,
        save        = False,
    )
    result = results[0]

    # Timing (ms)
    speed = result.speed or {}
    infer_times_ms.append(float(speed.get("inference", 0.0)))

    # Image dimensions
    h, w = result.orig_shape[:2]
    stem  = image_path.stem
    image_shapes[stem] = (h, w)

    # Build label lines and collect per-detection info
    label_lines    = []
    stem_instances = []

    if result.boxes is not None and len(result.boxes) and result.masks is not None:
        cls_ids = result.boxes.cls.cpu().numpy().astype(int)
        confs   = result.boxes.conf.cpu().numpy().astype(float)
        polys   = result.masks.xy          # list of (N, 2) pixel-space arrays

        for cls_id, conf_score, poly in zip(cls_ids, confs, polys):
            if poly is None or len(poly) < 3:
                skipped.append((image_path.name, int(cls_id)))
                continue
            # Normalise to [0, 1]
            pts_norm = poly / np.array([w, h], dtype=np.float32)
            pts_norm = np.clip(pts_norm, 0.0, 1.0)
            coords_str = " ".join(f"{x:.6f} {y:.6f}" for x, y in pts_norm)
            label_lines.append(f"{int(cls_id)} {coords_str}")

            # Store for metrics
            stem_instances.append({
                "cls":       int(cls_id),
                "conf":      float(conf_score),
                "poly_norm": pts_norm.tolist(),
            })

    pred_instances[stem] = stem_instances

    # Write .txt — empty file if no detections
    label_path = PRED_LABELS_DIR / (image_path.stem + ".txt")
    with label_path.open("w", encoding="utf-8") as f:
        f.write("\n".join(label_lines))

# --- Timing summary ---
infer_arr    = np.array(infer_times_ms)
time_mean_ms = float(np.mean(infer_arr))
time_std_ms  = float(np.std(infer_arr))
time_min_ms  = float(np.min(infer_arr))
time_max_ms  = float(np.max(infer_arr))

print(f"\nInference timing (net inference stage, ms/image):")
print(f"  mean : {time_mean_ms:.2f} ms")
print(f"  std  : {time_std_ms:.2f} ms")
print(f"  min  : {time_min_ms:.2f} ms")
print(f"  max  : {time_max_ms:.2f} ms")
print(f"  ~FPS : {1000/time_mean_ms:.1f}")

if skipped:
    print(f"\nWARNING: {len(skipped)} polygon(s) had < 3 points and were skipped:")
    for name, cid in skipped[:10]:
        print(f"  {name}  class={cid}")

# Save timing CSV
timing_summary = {
    "num_images":     len(all_image_paths),
    "infer_mean_ms":  round(time_mean_ms, 4),
    "infer_std_ms":   round(time_std_ms,  4),
    "infer_min_ms":   round(time_min_ms,  4),
    "infer_max_ms":   round(time_max_ms,  4),
    "fps_approx":     round(1000 / time_mean_ms, 2),
    "conf_threshold": INFER_CONF,
    "iou_threshold":  INFER_IOU,
    "weights":        str(DEPLOY_WEIGHTS_BEST),
}
timing_csv = RESULTS_DIR / f"{PROJECT_NAME}_inference_timing.csv"
pd.DataFrame([timing_summary]).to_csv(timing_csv, index=False)
print(f"\nTiming summary saved to: {timing_csv}")
print(f"Predicted labels saved to: {PRED_LABELS_DIR}  ({len(all_image_paths)} files)")


# --- Per-image per-region segmentation metrics ---
# Per class: rasterise GT and predicted polygons, compute pairwise mask IoU and match
# greedily by descending IoU (as Ultralytics match_predictions, use_scipy=False).
# Outputs: PER_IMAGE_CSV (one row per region instance) and PER_OBJECT_CSV.

print("\n" + "=" * 72)
print("Computing per-image per-region segmentation metrics …")
print("=" * 72)

# --- Helper: parse YOLO segmentation label file ---
def _parse_label_file(txt_path, img_w, img_h):
    """Parse one YOLO-seg .txt file into a list of {class_id, polygon_px}."""
    instances = []
    if not txt_path.is_file():
        return instances
    with txt_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:          # class_id + ≥ 3 xy pairs
                continue
            cls_id = int(float(parts[0]))
            coords = [float(v) for v in parts[1:]]
            if len(coords) % 2 != 0:
                coords = coords[:-1]
            poly_norm = list(zip(coords[0::2], coords[1::2]))
            poly_px   = [
                (int(round(x * img_w)), int(round(y * img_h)))
                for x, y in poly_norm
            ]
            instances.append({
                "class_id":   cls_id,
                "polygon_px": poly_px,
            })
    return instances


# --- Helper: rasterise polygon to boolean mask (cv2.fillPoly) ---
def _rasterise(poly_px, h, w):
    """Rasterise pixel-coordinate polygon to (H, W) boolean mask."""
    mask = np.zeros((h, w), dtype=np.uint8)
    if len(poly_px) < 3:
        return mask.astype(bool)
    pts = np.array(poly_px, dtype=np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(mask, [pts], 1)
    return mask.astype(bool)


# --- Helper: mask IoU matrix (NumPy version of Ultralytics mask_iou) ---
def _mask_iou_matrix(masks_a, masks_b, eps=1e-7):
    """
    Pairwise mask IoU.

    masks_a : list of (H, W) bool arrays  (GT)
    masks_b : list of (H, W) bool arrays  (pred)
    Returns : (N_a, N_b) float64 IoU matrix
    """
    na, nb = len(masks_a), len(masks_b)
    if na == 0 or nb == 0:
        return np.zeros((na, nb), dtype=np.float64)
    # Flatten to (N, H*W) float vectors
    a = np.stack([m.ravel().astype(np.float64) for m in masks_a])   # (na, H*W)
    b = np.stack([m.ravel().astype(np.float64) for m in masks_b])   # (nb, H*W)
    intersection = a @ b.T                            # (na, nb)
    area_a = a.sum(axis=1, keepdims=True)             # (na, 1)
    area_b = b.sum(axis=1, keepdims=True).T           # (1, nb)
    union  = area_a + area_b - intersection
    return intersection / (union + eps)


# --- Helper: greedy matching (as BaseValidator.match_predictions) ---
def _greedy_match(iou_matrix, threshold):
    """
    Greedy 1-to-1 matching within a single class.

    Iterates over all (gt, pred) pairs with IoU ≥ threshold in
    descending IoU order; the first uncontested match wins.
    Same procedure as Ultralytics match_predictions (use_scipy=False).

    Returns list of (gt_idx, pred_idx, iou_value).
    """
    if iou_matrix.size == 0:
        return []
    gt_idxs, pred_idxs = np.where(iou_matrix >= threshold)
    if len(gt_idxs) == 0:
        return []
    ious  = iou_matrix[gt_idxs, pred_idxs]
    order = np.argsort(-ious)           # descending IoU

    matches      = []
    matched_gt   = set()
    matched_pred = set()
    for idx in order:
        gi, pi = int(gt_idxs[idx]), int(pred_idxs[idx])
        if gi not in matched_gt and pi not in matched_pred:
            matches.append((gi, pi, float(ious[idx])))
            matched_gt.add(gi)
            matched_pred.add(pi)
    return matches


# --- Helper: mask centroid ---
def _mask_centroid(mask):
    """Return (cx, cy) of a boolean mask, or (NaN, NaN) if empty."""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return (float("nan"), float("nan"))
    return (float(np.mean(xs)), float(np.mean(ys)))


# --- Main loop: per-image, per-class matching ---
per_image_rows = []

for image_path in all_image_paths:
    stem = image_path.stem
    h, w = image_shapes[stem]

    # --- Parse GT labels from LABELS_DIR ---
    gt_txt  = LABELS_DIR / (stem + ".txt")
    gt_raw  = _parse_label_file(gt_txt, w, h)

    # --- Pred instances (collected during inference above) ---
    pr_raw  = pred_instances.get(stem, [])

    # --- Parse filename for object metadata ---
    try:
        material, object_id, perspective_id, _ = parse_material_object_perspective(
            image_path.name
        )
    except ValueError:
        material, object_id, perspective_id = "UNKNOWN", "UNKNOWN", "UNKNOWN"

    # --- Process each class independently ---
    for cls_id, cls_name in CLASS_ID_TO_NAME.items():
        # Gather GT instances for this class
        gt_class = [inst for inst in gt_raw if inst["class_id"] == cls_id]
        # Gather pred instances for this class
        pr_class = [inst for inst in pr_raw if inst["cls"] == cls_id]

        # Rasterise masks
        gt_masks = [_rasterise(inst["polygon_px"], h, w) for inst in gt_class]
        pr_masks = [_rasterise(
            [(int(round(x * w)), int(round(y * h))) for x, y in inst["poly_norm"]]
            if "poly_norm" in inst else inst["polygon_px"],
            h, w
        ) for inst in pr_class]

        # Precompute areas and centroids
        gt_areas     = [int(m.sum()) for m in gt_masks]
        pr_areas     = [int(m.sum()) for m in pr_masks]
        gt_centroids = [_mask_centroid(m) for m in gt_masks]
        pr_centroids = [_mask_centroid(m) for m in pr_masks]
        pr_confs     = [inst["conf"] for inst in pr_class]

        # IoU matrix and greedy matching
        iou_mat = _mask_iou_matrix(gt_masks, pr_masks)
        matches = _greedy_match(iou_mat, MATCH_IOU_THRESHOLD)

        matched_gt_set   = {gi for gi, _, _ in matches}
        matched_pred_set = {pi for _, pi, _ in matches}

        # --- TP rows (matched pairs) ---
        for gi, pi, iou_val in matches:
            gcx, gcy = gt_centroids[gi]
            pcx, pcy = pr_centroids[pi]
            cdist = float(np.sqrt((gcx - pcx)**2 + (gcy - pcy)**2))
            per_image_rows.append({
                "image_stem":      stem,
                "material":        material,
                "object_id":       object_id,
                "perspective_id":  perspective_id,
                "class_id":        cls_id,
                "class_name":      cls_name,
                "match_status":    "TP",
                "iou":             round(iou_val, 6),
                "gt_area_px":      gt_areas[gi],
                "pred_area_px":    pr_areas[pi],
                "gt_centroid_x":   round(gcx, 2),
                "gt_centroid_y":   round(gcy, 2),
                "pred_centroid_x": round(pcx, 2),
                "pred_centroid_y": round(pcy, 2),
                "centroid_dist_px": round(cdist, 2),
                "confidence":      round(pr_confs[pi], 6),
            })

        # --- FN rows (unmatched GT) ---
        for gi in range(len(gt_class)):
            if gi in matched_gt_set:
                continue
            gcx, gcy = gt_centroids[gi]
            per_image_rows.append({
                "image_stem":      stem,
                "material":        material,
                "object_id":       object_id,
                "perspective_id":  perspective_id,
                "class_id":        cls_id,
                "class_name":      cls_name,
                "match_status":    "FN",
                "iou":             float("nan"),
                "gt_area_px":      gt_areas[gi],
                "pred_area_px":    float("nan"),
                "gt_centroid_x":   round(gcx, 2),
                "gt_centroid_y":   round(gcy, 2),
                "pred_centroid_x": float("nan"),
                "pred_centroid_y": float("nan"),
                "centroid_dist_px": float("nan"),
                "confidence":      float("nan"),
            })

        # --- FP rows (unmatched predictions) ---
        for pi in range(len(pr_class)):
            if pi in matched_pred_set:
                continue
            pcx, pcy = pr_centroids[pi]
            per_image_rows.append({
                "image_stem":      stem,
                "material":        material,
                "object_id":       object_id,
                "perspective_id":  perspective_id,
                "class_id":        cls_id,
                "class_name":      cls_name,
                "match_status":    "FP",
                "iou":             float("nan"),
                "gt_area_px":      float("nan"),
                "pred_area_px":    pr_areas[pi],
                "gt_centroid_x":   float("nan"),
                "gt_centroid_y":   float("nan"),
                "pred_centroid_x": round(pcx, 2),
                "pred_centroid_y": round(pcy, 2),
                "centroid_dist_px": float("nan"),
                "confidence":      round(pr_confs[pi], 6),
            })

# --- Per-image DataFrame and CSV ---
df_per_image = pd.DataFrame(per_image_rows)

if df_per_image.empty:
    print("WARNING: no instances found — CSVs will be empty.")
    df_per_image.to_csv(PER_IMAGE_CSV, index=False)
    pd.DataFrame().to_csv(PER_OBJECT_CSV, index=False)
else:
    df_per_image.to_csv(PER_IMAGE_CSV, index=False)
    print(f"\nPer-image region metrics saved to: {PER_IMAGE_CSV}")
    print(f"  Total rows:  {len(df_per_image)}")
    print(f"  TP: {(df_per_image['match_status'] == 'TP').sum()}  "
          f"FP: {(df_per_image['match_status'] == 'FP').sum()}  "
          f"FN: {(df_per_image['match_status'] == 'FN').sum()}")

    # --- Per-class match summary ---
    print("\nPer-class match summary:")
    summary = (
        df_per_image
        .groupby(["class_name", "match_status"])
        .size()
        .unstack(fill_value=0)
    )
    display(summary)

    tp_df = df_per_image[df_per_image["match_status"] == "TP"]
    if not tp_df.empty:
        print("\nTP IoU statistics per class:")
        display(
            tp_df.groupby("class_name")["iou"]
            .agg(["count", "mean", "std", "min", "max"])
            .round(4)
        )

    # --- Per-object aggregation ---
    # Grouped by (material, object_id, class_name); only perspectives with GT
    # instances of the class contribute (a missing class is not a zero score).

    agg_rows = []
    group_cols = ["material", "object_id", "class_name", "class_id"]

    # Identify perspective-level GT presence
    gt_presence = (
        df_per_image[df_per_image["match_status"].isin(["TP", "FN"])]
        .drop_duplicates(subset=["material", "object_id", "perspective_id", "class_name"])
    )

    for (mat, oid, cname, cid), grp in df_per_image.groupby(group_cols):
        tp = grp[grp["match_status"] == "TP"]
        fn = grp[grp["match_status"] == "FN"]
        fp = grp[grp["match_status"] == "FP"]

        n_tp = len(tp)
        n_fn = len(fn)
        n_fp = len(fp)

        # Number of perspectives where this class has GT
        n_perspectives_with_gt = len(
            gt_presence[
                (gt_presence["material"] == mat)
                & (gt_presence["object_id"] == oid)
                & (gt_presence["class_name"] == cname)
            ]["perspective_id"].unique()
        )

        # IoU stats over TP matches only
        if n_tp > 0:
            iou_vals = tp["iou"].values
            iou_mean = round(float(np.nanmean(iou_vals)), 6)
            iou_std  = round(float(np.nanstd(iou_vals)),  6)
            iou_min  = round(float(np.nanmin(iou_vals)),  6)
            iou_max  = round(float(np.nanmax(iou_vals)),  6)
        else:
            iou_mean = iou_std = iou_min = iou_max = float("nan")

        # Detection rate / precision
        recall    = round(n_tp / (n_tp + n_fn), 6) if (n_tp + n_fn) > 0 else float("nan")
        precision = round(n_tp / (n_tp + n_fp), 6) if (n_tp + n_fp) > 0 else float("nan")
        f1 = (
            round(2 * precision * recall / (precision + recall), 6)
            if (precision + recall) > 0 and not (np.isnan(precision) or np.isnan(recall))
            else float("nan")
        )

        # Mean centroid distance (TP only)
        cd_mean = (
            round(float(tp["centroid_dist_px"].mean()), 2)
            if n_tp > 0 else float("nan")
        )

        agg_rows.append({
            "material":               mat,
            "object_id":              oid,
            "class_id":               cid,
            "class_name":             cname,
            "n_perspectives_with_gt": n_perspectives_with_gt,
            "n_tp":                   n_tp,
            "n_fp":                   n_fp,
            "n_fn":                   n_fn,
            "recall":                 recall,
            "precision":              precision,
            "f1":                     f1,
            "iou_mean":               iou_mean,
            "iou_std":                iou_std,
            "iou_min":                iou_min,
            "iou_max":                iou_max,
            "centroid_dist_mean_px":  cd_mean,
        })

    df_per_object = pd.DataFrame(agg_rows).sort_values(
        ["material", "object_id", "class_id"]
    ).reset_index(drop=True)

    df_per_object.to_csv(PER_OBJECT_CSV, index=False)

    print(f"\nPer-object region metrics saved to: {PER_OBJECT_CSV}")
    print(f"  Total rows: {len(df_per_object)}")
    print(f"  Match IoU threshold: {MATCH_IOU_THRESHOLD}")

    print("\nPer-object aggregation — head:")
    display(df_per_object.head(12))

    # Class-level summary across all objects
    print("\nGrand summary per class (across all objects):")
    grand = df_per_object.groupby("class_name").agg(
        n_objects         = ("object_id", "nunique"),
        total_tp          = ("n_tp",  "sum"),
        total_fp          = ("n_fp",  "sum"),
        total_fn          = ("n_fn",  "sum"),
        mean_iou_of_tp    = ("iou_mean", "mean"),
        mean_recall       = ("recall", "mean"),
        mean_precision    = ("precision", "mean"),
    ).round(4)
    display(grand)

In [ ]:
# === Cell 10: Inference Speed Timing ===
# Times predict() on all images after warm-up passes, with CUDA synchronisation around
# each call, and writes one CSV row per measurement (image x repeat, warm-up flagged).
# Needs only the cells up to the Object-Level Train/Val Split; weights come from disk.

import time  # not in the Imports cell

# --- Timing config ---
TIMING_N_WARMUP   = 30     # discarded passes to reach steady state (flagged in CSV)
TIMING_N_REPEATS  = 5      # timed passes per image (>=1); set to 1 for one pass/image
TIMING_SORT_IMAGES = True  # deterministic image order

# Inference settings, same as in the Full-Dataset Inference and Region Metrics cell
TIMING_INFER_CONF    = 0.50
TIMING_INFER_IOU     = 0.50
TIMING_INFER_MAX_DET = 300
TIMING_INFER_RETINA  = True

RAW_TIMING_CSV_BASE = RESULTS_DIR / f"{PROJECT_NAME}_inference_timing_raw.csv"

# --- Helper: never overwrite an existing result file ---
def _unique_path(base_path):
    """Return base_path if free, else the same stem with a _YYYYmmdd_HHMMSS tag."""
    if not base_path.exists():
        return base_path
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return base_path.with_name(f"{base_path.stem}_{stamp}{base_path.suffix}")

RAW_TIMING_CSV = _unique_path(RAW_TIMING_CSV_BASE)

# --- Collect images ---
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}
timing_image_paths = [
    p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS
]
if TIMING_SORT_IMAGES:
    timing_image_paths = sorted(timing_image_paths)
if not timing_image_paths:
    raise FileNotFoundError(f"No images found in {IMAGES_DIR}")
print(f"Images to time: {len(timing_image_paths)}")
print(f"Warm-up passes: {TIMING_N_WARMUP} | timed repeats/image: {TIMING_N_REPEATS}")

# --- Resolve deployment weights (same order as in the Full-Dataset Inference cell) ---
weights_source = None

if "DEPLOY_WEIGHTS_BEST" in globals() and Path(DEPLOY_WEIGHTS_BEST).exists():
    DEPLOY_WEIGHTS_BEST = Path(DEPLOY_WEIGHTS_BEST)
    weights_source = "in-memory (Cell 7 ran this session)"

elif FINAL_RUN_SUMMARY_JSON.exists():
    with FINAL_RUN_SUMMARY_JSON.open("r", encoding="utf-8") as f:
        _run_summary = json.load(f)
    _candidate = Path(_run_summary.get("weights_best", ""))
    if _candidate.exists():
        DEPLOY_WEIGHTS_BEST = _candidate
        weights_source = f"run summary JSON: {FINAL_RUN_SUMMARY_JSON.name}"

if weights_source is None:
    _candidate = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
    if _candidate.exists():
        DEPLOY_WEIGHTS_BEST = _candidate
        weights_source = f"deterministic path: {RUN_NAME}/weights/best.pt"

if weights_source is None:
    _matches = sorted(
        RUNS_DIR.glob(f"{RUN_NAME}*/weights/best.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if _matches:
        DEPLOY_WEIGHTS_BEST = _matches[0]
        weights_source = f"most recent matching run folder: {DEPLOY_WEIGHTS_BEST.parent.parent.name}"

if weights_source is None:
    raise FileNotFoundError(
        "Could not locate deployment weights automatically. "
        "Run Cell 7 first, or set DEPLOY_WEIGHTS_BEST manually."
    )

print(f"Using weights ({weights_source}):")
print(f"  {DEPLOY_WEIGHTS_BEST}")

timing_model = YOLO(str(DEPLOY_WEIGHTS_BEST))

# --- Timing primitives ---
_cuda = torch.cuda.is_available()
_sync_dev = DEVICE if isinstance(DEVICE, int) else None

def _cuda_sync():
    if _cuda:
        torch.cuda.synchronize(_sync_dev)

def _timed_predict(image_path):
    """One synchronised predict() call. Returns (speed_dict, wall_ms, n_det)."""
    _cuda_sync()
    t0 = time.perf_counter()
    results = timing_model.predict(
        source       = str(image_path),
        conf         = TIMING_INFER_CONF,
        iou          = TIMING_INFER_IOU,
        imgsz        = IMAGE_SIZE,
        max_det      = TIMING_INFER_MAX_DET,
        retina_masks = TIMING_INFER_RETINA,
        device       = DEVICE,
        verbose      = False,
        save         = False,
        save_txt     = False,
        save_conf    = False,
    )
    _cuda_sync()
    wall_ms = (time.perf_counter() - t0) * 1000.0
    result  = results[0]
    speed   = result.speed or {}
    n_det   = int(len(result.boxes)) if result.boxes is not None else 0
    return speed, wall_ms, n_det

# --- Warm-up (recorded and flagged, excluded from the summary) ---
timing_rows = []
global_index = 0

print("\nWarming up …")
for w in range(TIMING_N_WARMUP):
    img = timing_image_paths[w % len(timing_image_paths)]
    speed, wall_ms, n_det = _timed_predict(img)
    timing_rows.append({
        "global_index":   global_index,
        "image_index":    -1,
        "image_name":     img.name,
        "repeat":         w,
        "is_warmup":      True,
        "preprocess_ms":  round(float(speed.get("preprocess", float("nan"))), 6),
        "inference_ms":   round(float(speed.get("inference",  float("nan"))), 6),
        "postprocess_ms": round(float(speed.get("postprocess", float("nan"))), 6),
        "wall_ms":        round(wall_ms, 6),
        "n_detections":   n_det,
    })
    global_index += 1

# --- Timed run: per image x repeat ---
print("Timing …")
for image_index, image_path in enumerate(timing_image_paths):
    for rep in range(TIMING_N_REPEATS):
        speed, wall_ms, n_det = _timed_predict(image_path)
        timing_rows.append({
            "global_index":   global_index,
            "image_index":    image_index,
            "image_name":     image_path.name,
            "repeat":         rep,
            "is_warmup":      False,
            "preprocess_ms":  round(float(speed.get("preprocess", float("nan"))), 6),
            "inference_ms":   round(float(speed.get("inference",  float("nan"))), 6),
            "postprocess_ms": round(float(speed.get("postprocess", float("nan"))), 6),
            "wall_ms":        round(wall_ms, 6),
            "n_detections":   n_det,
        })
        global_index += 1

df_timing_raw = pd.DataFrame(timing_rows)

# --- Save raw CSV ---
df_timing_raw.to_csv(RAW_TIMING_CSV, index=False)
print(f"\nRaw per-measurement timing saved to: {RAW_TIMING_CSV}")
if RAW_TIMING_CSV != RAW_TIMING_CSV_BASE:
    print(f"  (base name existed; wrote timestamped file to avoid overwrite)")
print(f"  Rows: {len(df_timing_raw)}  "
      f"(warm-up: {int(df_timing_raw['is_warmup'].sum())}, "
      f"timed: {int((~df_timing_raw['is_warmup']).sum())})")

# --- Printed summary (timed rows only, no file written) ---
# Ultralytics net-inference stage and CUDA-synchronised wall time.
timed = df_timing_raw[~df_timing_raw["is_warmup"]]

def _describe(series, label):
    v = series.to_numpy(dtype=float)
    print(f"\n{label} (ms/image, {len(v)} timed passes):")
    print(f"  mean   : {np.mean(v):.4f}")
    print(f"  std    : {np.std(v):.4f}")
    print(f"  median : {np.median(v):.4f}")
    print(f"  min    : {np.min(v):.4f}")
    print(f"  p95    : {np.percentile(v, 95):.4f}")
    print(f"  max    : {np.max(v):.4f}")
    print(f"  ~FPS (from mean)   : {1000.0/np.mean(v):.1f}")
    print(f"  ~FPS (from median) : {1000.0/np.median(v):.1f}")

print("\n" + "=" * 72)
print("Inference speed summary (timed passes only, warm-up excluded)")
print("=" * 72)
_describe(timed["inference_ms"], "Net inference stage (Ultralytics result.speed['inference'])")
_describe(timed["wall_ms"],      "End-to-end wall time (CUDA-synced predict() call)")

# Per-image steady-state latency: minimum over repeats
per_image_min = (
    timed.groupby("image_name")["inference_ms"].min().to_numpy(dtype=float)
)
print(f"\nPer-image best-of-{TIMING_N_REPEATS} inference latency:")
print(f"  images        : {len(per_image_min)}")
print(f"  mean of mins  : {np.mean(per_image_min):.4f} ms")
print(f"  max of mins   : {np.max(per_image_min):.4f} ms   "
      f"(slowest image at steady state)")
print(f"  ~FPS          : {1000.0/np.mean(per_image_min):.1f}")